# Laubmann-KG — full workflow (all 34 volumes, Colab)

One notebook, three stages, all reading/writing your Drive:

| stage | what | output |
|---|---|---|
| **A** | build the text + multimodal corpus from `Laubmann_NN_gemini/` region JSONs | `corpus_<date>/` |
| **B** | detect duplicate pages → human review → apply decisions | `corpus_<date>_dedup/` |
| **C** | LLM observation extraction → RDF/JSON-LD + SHACL + DwC-A for **all 34 volumes** | `exports_full/` |

Stage B has a **human step in the middle**: download `review.html`, adjudicate the
clusters, export `dedup_decisions.json`, upload it back, then continue.
Each stage is skippable if its output already exists — re-running the notebook
after a disconnect is safe (LLM calls are cached on Drive).

## 0 · Mount Drive, clone repo, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/laubmann-kg_TP'
if not os.path.isdir(REPO):
    !git clone https://github.com/Maelkolb/laubmann-kg_TP.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}

!pip -q install -e ".[llm]"
!pip -q install rapidfuzz datasketch imagehash pillow pandas

ADDONS = f'{REPO}/HistOrniGraph_addons'
DEDUP  = f'{ADDONS}/dedup'
assert os.path.isdir(f'{ADDONS}/laubmann_corpus'), 'addons missing — repo out of date?'
print('repo + addons ready')

## 1 · Config — edit these

In [ ]:
from pathlib import Path

# Root on Drive holding the Laubmann_NN_gemini/ folders (each with regions/*.json):
OUTPUT_BASE  = Path('/content/drive/MyDrive/HistOrniGraph_output')

# The corpus this run works on (stage A writes it, stages B/C read it):
CORPUS_DIR   = OUTPUT_BASE / 'corpus_2026-07-21'
CORPUS_DEDUP = OUTPUT_BASE / (CORPUS_DIR.name + '_dedup')

# Where stage C writes the knowledge-graph exports (on Drive, survives disconnects):
EXPORTS_DIR  = OUTPUT_BASE / 'kg_exports_full'

# LLM cache on Drive: interrupted extraction resumes for free.
LLM_CACHE    = OUTPUT_BASE / 'llm_cache'

# Diary volumes only — vol 35 is the general index/register, keep it excluded.
VOLUMES = list(range(1, 35))

for k, v in [('OUTPUT_BASE', OUTPUT_BASE), ('CORPUS_DIR', CORPUS_DIR),
             ('CORPUS_DEDUP', CORPUS_DEDUP), ('EXPORTS_DIR', EXPORTS_DIR)]:
    print(f'{k:13s}', v, '✓ exists' if v.exists() else '(will be created)')

In [ ]:
# Gemini key from Colab secrets (Runtime ▸ Secrets ▸ add LST_Gemini)
import os
from google.colab import userdata
key = userdata.get('LST_Gemini')
os.environ['GOOGLE_API_KEY'] = key
os.environ['GEMINI_API_KEY'] = key
print('Gemini key loaded:', bool(key))

## Stage A · Build the corpus (skip if `CORPUS_DIR` already exists)

Standard-library only; ~34 volumes take a few minutes of Drive I/O.

In [ ]:
if (CORPUS_DIR / 'corpus.json').exists():
    print('corpus already built at', CORPUS_DIR, '— skipping stage A')
else:
    vols = ' '.join(map(str, VOLUMES))
    !cd "{ADDONS}" && python build_corpus.py \
        --output-base "{OUTPUT_BASE}" \
        --corpus-dir  "{CORPUS_DIR}" \
        --per-volume --multimodal --report --volumes {vols}

In [ ]:
# Sanity: size + the known Vol-1→Vol-15 cross-run contamination check
import json
from collections import defaultdict
pages = json.load(open(CORPUS_DIR / 'corpus.json'))
print(len(pages), 'pages')

by_pid = defaultdict(set)
for p in pages:
    by_pid[p['page_id']].add(int(p['volume']))
cross = {pid: sorted(v) for pid, v in by_pid.items() if len(v) > 1}
print('page_ids appearing in >1 volume:', len(cross),
      '→ dedup will cluster these' if cross else '→ no cross-volume contamination')

## Stage B1 · Detect duplicates + build the review GUI

In [ ]:
DEDUP_OUT = CORPUS_DIR / 'dedup'

!cd "{DEDUP}" && python detect_duplicates.py "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}" \
    --scan-window 3 --cluster-threshold 0.55 --high-threshold 0.80 \
    --image-root "{OUTPUT_BASE}"

!cd "{DEDUP}" && python build_review_gui.py "{DEDUP_OUT}/duplicates_report.jsonl" \
    --corpus "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}/review.html"

import pandas as pd
dup = pd.read_csv(DEDUP_OUT / 'duplicates_report.csv')
print(len(dup), 'clusters flagged —', (dup.suggested_action == 'drop_duplicates').sum(), 'auto-drop (≥0.80),',
      (dup.suggested_action != 'drop_duplicates').sum(), 'need review')
dup.head(10)

In [ ]:
# Download review.html → adjudicate → the GUI exports dedup_decisions.json.
# (It is also on Drive at DEDUP_OUT/review.html if the download times out.)
from google.colab import files
files.download(str(DEDUP_OUT / 'review.html'))

## Stage B2 · Apply decisions → clean corpus

Put the exported `dedup_decisions.json` into `CORPUS_DIR/dedup/` on Drive
(or upload it here). Non-destructive: writes a NEW `corpus_*_dedup/` with
regenerated `entries.csv` (incl. `entry_uid`/`page_uid`/`region_uid`) and a
`dedup_manifest` recording every dropped page.

In [ ]:
DECISIONS = DEDUP_OUT / 'dedup_decisions.json'
if not DECISIONS.exists():
    from google.colab import files
    up = files.upload()                      # pick dedup_decisions.json
    (DEDUP_OUT).mkdir(parents=True, exist_ok=True)
    Path(DECISIONS).write_bytes(next(iter(up.values())))

!python "{DEDUP}/apply_dedup.py" \
    --corpus-dir "{CORPUS_DIR}" \
    --decisions  "{DECISIONS}" \
    --out-dir    "{CORPUS_DEDUP}"

import pandas as pd
assert (CORPUS_DEDUP / 'entries.csv').exists(), 'entries not regenerated'
ent = pd.read_csv(CORPUS_DEDUP / 'entries.csv')
man = pd.read_csv(CORPUS_DEDUP / 'dedup_manifest.csv')
assert ent.entry_uid.notna().all() and ent.entry_uid.is_unique, 'entry_uid must be unique'
print(len(man), 'pages dropped;', len(ent), 'entries in the deduped corpus')
print(ent.groupby('volume').size())

In [ ]:
# Multimodal catalogue: copy it beside the deduped entries so --input-dir finds it.
import shutil
src = CORPUS_DIR / 'multimodal' / 'multimodal.md'
if src.exists():
    shutil.copy(src, CORPUS_DEDUP / 'multimodal.md')
    print('multimodal.md copied into', CORPUS_DEDUP)
else:
    print('no multimodal.md found — DwC-A will simply have no multimedia rows')

## Stage C · Knowledge graph for all 34 volumes (Gemini)

`configs/full_llm.yaml` has `sample.volume: null` (= all volumes) and QA on.
With ~11k entries this is the long stage — hours, not minutes. The LLM cache
lives on Drive, so an interrupted run resumes where it stopped: just re-run
the cell.

In [ ]:
# Put the LLM cache on Drive so an interrupted run resumes across sessions
import yaml
LLM_CACHE.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load(open('configs/full_llm.yaml'))
cfg['extraction']['cache_dir'] = str(LLM_CACHE)
yaml.safe_dump(cfg, open('configs/full_llm.yaml', 'w'), sort_keys=False, allow_unicode=True)
print('LLM cache →', LLM_CACHE)

In [ ]:
!laubmann-kg export-jsonld --config configs/full_llm.yaml \
    --input-dir "{CORPUS_DEDUP}" --output-dir "{EXPORTS_DIR}" 

In [ ]:
!laubmann-kg export-dwca --config configs/full_llm.yaml \
    --input-dir "{CORPUS_DEDUP}" --output-dir "{EXPORTS_DIR}" 

## 3 · Check the graph — competency questions + metadata coverage

In [ ]:
import sys
sys.path.insert(0, 'src')
from laubmann_kg.kg.sparql import load_graph, run_all
from rdflib import RDF
from laubmann_kg.kg.rdf import LKG
import pandas as pd

g = load_graph(str(EXPORTS_DIR / 'rdf' / 'laubmann_sample.ttl'))
print('Triples:', len(g))
print('Entries:', len(set(g.subjects(RDF.type, LKG.DiaryEntry))))
print('Observations:', len(set(g.subjects(RDF.type, LKG.ObservationEvent))), '\n')

for cq, rows in run_all(g).items():
    print(f'{cq}: {len(rows)} rows')

In [ ]:
# Metadata field coverage: how full is each property, per volume?
from rdflib import RDF
obs = list(g.subjects(RDF.type, LKG.ObservationEvent))
def frac(pred):
    return sum(1 for o in obs if g.value(o, pred) is not None) / max(len(obs), 1)
for name in ['observedTaxon', 'observedAt', 'individualCount', 'countQualifier',
             'hasEvidence', 'verbatimNotes', 'derivedFromEntry']:
    print(f'{name:18s} {frac(LKG[name]):6.1%}')

## 4 · Bundle the exports

In [ ]:
# Everything is already on Drive under EXPORTS_DIR; zip if you want a download.
!cd "{EXPORTS_DIR}/.." && zip -qr kg_exports_full.zip "{EXPORTS_DIR.name}"
print('zip at', EXPORTS_DIR.parent / 'kg_exports_full.zip')